<a href="https://www.kaggle.com/code/laymond/s6e8-elasticnet-lb0-97108-cv0-9700298?scriptVersionId=348577629" target="_blank"><img align="left" alt="Kaggle" title="Open in Kaggle" src="https://kaggle.com/static/images/open-in-kaggle.svg"></a>

# S6E8 — Elastic Net Stacker

Builds a rank-transformed stacking ensemble from two out-of-fold (OOF)
prediction libraries:

- **Dataset A** — 15 models/blends (CSV predictions)
- **Dataset B** — 74 models (`.npy` predictions)

An Elastic Net logistic regression is fit as the meta-learner on top of
the rank-transformed OOF predictions, validated with 5-fold CV, then
refit on all data to produce the final submission.

## Data Sources

| Author | Dataset |
|----------|----------|
| Szymon Kłapiński| [S6E8 full OOF library](https://www.kaggle.com/datasets/szymonkapiski/s6e8-oof-library-47-models) |
| Naji | [Playground S6E8 - OOF & Submission](https://www.kaggle.com/datasets/najiama/predicting-smartphone-addiction-oof-submission-csv)  |


## Setup

Paths, file lists, and the training target.

In [ ]:
import os
import numpy as np
import pandas as pd

train_path = "/kaggle/input/competitions/playground-series-s6e8/train.csv"
test_path = "/kaggle/input/competitions/playground-series-s6e8/test.csv"  

A_DIR = "/kaggle/input/datasets/najiama/predicting-smartphone-addiction-oof-submission-csv"
B_DIR = "/kaggle/input/datasets/szymonkapiski/s6e8-oof-library-47-models/oof"

train = pd.read_csv(train_path)
test_df = pd.read_csv(test_path)  

y = train["addicted_label"].to_numpy()  

a_oof_files = sorted(f for f in os.listdir(A_DIR) if f.endswith("_oof_predictions.csv"))  
b_oof_files = sorted(f for f in os.listdir(B_DIR) if f.startswith("oof_") and f.endswith(".npy"))  

print(f"Dataset A OOF files: {len(a_oof_files)}")
print(f"Dataset B OOF files: {len(b_oof_files)}")

## Build the OOF feature matrix

Stack Dataset A's 15 CSV predictions and Dataset B's 74 `.npy`
predictions into a single `(n_rows, 89)` matrix, with matching
`feature_names` for later use on the test set.

In [ ]:
# Dataset A
A_cols = []
A_matrix = []

for f in a_oof_files:
    df = pd.read_csv(os.path.join(A_DIR, f))
    pred_col = [c for c in df.columns if c != "id"][0]
    x = df[pred_col].to_numpy(dtype=np.float64)

    assert len(x) == len(train), f"{f}: wrong length"
    assert np.isfinite(x).all(), f"{f}: contains NaN/inf"

    A_matrix.append(x)
    A_cols.append("A_" + f.replace(".csv", ""))

A_matrix = np.column_stack(A_matrix)

# Dataset B
B_cols = []
B_matrix = []

for f in b_oof_files:
    x = np.load(os.path.join(B_DIR, f)).astype(np.float64)

    assert x.shape == (len(train),), f"{f}: wrong shape {x.shape}"
    assert np.isfinite(x).all(), f"{f}: contains NaN/inf"

    B_matrix.append(x)
    B_cols.append("B_" + f.replace(".npy", ""))

B_matrix = np.column_stack(B_matrix)

# Combine
X_oof = np.column_stack([A_matrix, B_matrix])
feature_names = A_cols + B_cols

print("A matrix:", A_matrix.shape)
print("B matrix:", B_matrix.shape)
print("Full OOF matrix:", X_oof.shape)
print("Finite:", np.isfinite(X_oof).all())

## Rank-transform features

Convert each model's raw predictions to within-column percentile ranks
so every feature sits on a common `(0, 1)` scale, regardless of how
each source model was calibrated.

In [ ]:
from scipy.stats import rankdata

X_rank = np.empty_like(X_oof, dtype=np.float32)

for j in range(X_oof.shape[1]):
    X_rank[:, j] = rankdata(X_oof[:, j], method="average") / len(X_oof)

print("Rank matrix:", X_rank.shape, X_rank.dtype)
print("Range:", X_rank.min(), "-", X_rank.max())

## Cross-validate the Elastic Net stacker

5-fold stratified CV to get an honest out-of-fold AUC / log-loss for
the Elastic Net logistic regression meta-model before fitting on all
the data.

In [ ]:
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import log_loss, roc_auc_score
from sklearn.model_selection import StratifiedKFold

N_SPLITS = 5

skf = StratifiedKFold(n_splits=N_SPLITS, shuffle=True, random_state=42)

stack_oof = np.zeros(len(y), dtype=np.float64)
fold_coefs = []

for fold, (train_idx, valid_idx) in enumerate(skf.split(X_rank, y), 1):
    stacker = LogisticRegression(
        penalty="elasticnet",
        solver="saga",
        C=0.1,
        l1_ratio=0.5,
        max_iter=1000,
        tol=1e-4,
        random_state=42,
        n_jobs=-1,
    )

    stacker.fit(X_rank[train_idx], y[train_idx])
    stack_oof[valid_idx] = stacker.predict_proba(X_rank[valid_idx])[:, 1]
    fold_coefs.append(stacker.coef_[0].copy())

    fold_auc = roc_auc_score(y[valid_idx], stack_oof[valid_idx])
    fold_logloss = log_loss(y[valid_idx], stack_oof[valid_idx])
    print(f"Fold {fold}/{N_SPLITS} — AUC: {fold_auc:.7f}  LogLoss: {fold_logloss:.7f}")

cv_auc = roc_auc_score(y, stack_oof)
cv_logloss = log_loss(y, stack_oof)

print(f"\n5-Fold CV — AUC: {cv_auc:.7f}  LogLoss: {cv_logloss:.7f}")

## Fit the final stacker & build the test feature matrix

Refit the Elastic Net on **all** OOF rows, then assemble the test
matrix in the exact same feature order as `X_rank` (Dataset A
submission CSVs + Dataset B `test_*.npy` arrays).

In [ ]:
final_stacker = LogisticRegression(
    penalty="elasticnet",
    solver="saga",
    C=0.1,
    l1_ratio=0.5,
    max_iter=1000,
    tol=1e-4,
    random_state=42,
    n_jobs=-1,
)

print("Fitting final Elastic Net on all OOF rows...")
final_stacker.fit(X_rank, y)
print("Done.")

# Dataset A test predictions (submission CSVs)
A_TEST_DIR = A_DIR

A_test_mapping = {
    "A_01_oof_predictions": "01_submission.csv",
    "A_02_oof_predictions": "02_submission.csv",
    "A_03_oof_predictions": "03_submission.csv",
    "A_04_oof_predictions": "04_submission.csv",
    "A_05_oof_predictions": "05_submission.csv",
    "A_07_blend_oof_predictions": "07_blend_submission.csv",
    "A_08_blend_oof_predictions": "08_blend_submission.csv",
    "A_09_blend_oof_predictions": "09_blend_submission.csv",
    "A_10_blend_oof_predictions": "10_blend_submission.csv",
    "A_12_blend_oof_predictions": "12_blend_submission.csv",
    "A_13_blend_oof_predictions": "13_blend_submission.csv",
    "A_14_blend_oof_predictions": "14_blend_submission.csv",
    "A_16_blend_oof_predictions": "16_blend_submission.csv",
    "A_18_blend_oof_predictions": "18_blend_submission.csv",
    "A_19_blend_oof_predictions": "19_blend_submission.csv.csv",  
}

A_test = []
for feature_name, filename in A_test_mapping.items():
    path = os.path.join(A_TEST_DIR, filename)
    assert os.path.exists(path), f"Missing: {path}"

    df = pd.read_csv(path)
    assert len(df) == len(test_df), f"{filename}: expected {len(test_df)}, got {len(df)}"

    pred_cols = [c for c in df.columns if c != "id"]
    assert len(pred_cols) == 1, f"{filename}: expected one prediction column, found {pred_cols}"

    A_test.append(df[pred_cols[0]].to_numpy(dtype=np.float32))

A_test_dict = dict(zip(A_test_mapping.keys(), A_test))

# Dataset B test predictions (test_*.npy, alongside the oof_*.npy files)
B_TEST_DIR = B_DIR

B_test_dict = {}
for feature_name in feature_names:
    if not feature_name.startswith("B_oof_"):
        continue

    base = feature_name[len("B_oof_"):]
    path = os.path.join(B_TEST_DIR, f"test_{base}.npy")
    assert os.path.exists(path), f"Missing: {path}"

    arr = np.load(path)
    assert arr.shape[0] == len(test_df), f"{path}: expected {len(test_df)}, got {arr.shape}"

    B_test_dict[feature_name] = arr.astype(np.float32)

# Assemble in the exact same feature order as X_rank
X_test = np.column_stack([
    A_test_dict[name] if name.startswith("A_") else B_test_dict[name]
    for name in feature_names
]).astype(np.float32)

assert X_test.shape == (len(test_df), len(feature_names))
assert np.isfinite(X_test).all()

print("Test matrix:", X_test.shape)
print("Finite:", np.isfinite(X_test).all())

## Rank-transform the test set & generate the submission

Apply the same percentile-rank transform used on the OOF matrix, run
the final Elastic Net stacker, and write out `submission.csv`.

In [ ]:
X_test_rank = np.empty_like(X_test, dtype=np.float32)

for j in range(X_test.shape[1]):
    X_test_rank[:, j] = (
        rankdata(X_test[:, j], method="average") / len(X_test)
    ).astype(np.float32)

print("Test rank matrix:", X_test_rank.shape)

test_stack = final_stacker.predict_proba(X_test_rank)[:, 1]

print("Predictions — min:", test_stack.min(), "max:", test_stack.max(),
      "mean:", test_stack.mean(), "std:", test_stack.std())

submission = pd.DataFrame({
    "id": test_df["id"].values,
    "addicted_label": test_stack,
})

submission_path = "/kaggle/working/submission_elastic_net_final.csv"
submission.to_csv(submission_path, index=False)

assert submission["id"].is_unique
assert len(submission) == len(test_df)
assert np.isfinite(submission["addicted_label"]).all()

print("Submission created:", submission_path)
submission.head()